# Coco Crepe — 05 Silver Inventory

Transformación del Data Product de inventario consumiendo `gold.product_master`.

**Mejora aplicada:**
- Deduplicación de inventario por `product_id`, conservando la actualización más reciente.
- Deduplicación de proveedores por `supplier_id`.
- Filtrado de stock negativo y precios inválidos.

In [0]:
GROUP = "g203"

SALES_DATA_PRODUCT = "sales_summary"
INVENTORY_DATA_PRODUCT = "inventory_status"
PRODUCT_DATA_PRODUCT = "product_master"

# Completa estos valores solo si la detección automática no encuentra
# exactamente un catálogo por Data Product.
SALES_CATALOG_MANUAL = None
INVENTORY_CATALOG_MANUAL = "g203_inv_inventory_status"
PRODUCT_CATALOG_MANUAL = None

def resolve_catalog(data_product_name, manual_catalog=None):
    if manual_catalog:
        return manual_catalog

    catalogs = [row[0] for row in spark.sql("SHOW CATALOGS").collect()]
    target = data_product_name.lower()

    preferred = [
        catalog for catalog in catalogs
        if GROUP.lower() in catalog.lower()
        and target in catalog.lower()
    ]

    if len(preferred) == 1:
        return preferred[0]

    matches = [
        catalog for catalog in catalogs
        if target in catalog.lower()
    ]

    if len(matches) == 1:
        return matches[0]

    raise ValueError(
        f"No se pudo identificar un catálogo único para '{data_product_name}'. "
        f"Catálogos visibles: {catalogs}. "
        "Completa la variable *_CATALOG_MANUAL correspondiente."
    )

SALES_CATALOG = resolve_catalog(
    SALES_DATA_PRODUCT,
    SALES_CATALOG_MANUAL
)

INVENTORY_CATALOG = resolve_catalog(
    INVENTORY_DATA_PRODUCT,
    INVENTORY_CATALOG_MANUAL
)

PRODUCT_CATALOG = resolve_catalog(
    PRODUCT_DATA_PRODUCT,
    PRODUCT_CATALOG_MANUAL
)

print(f"Sales catalog: {SALES_CATALOG}")
print(f"Inventory catalog: {INVENTORY_CATALOG}")
print(f"Product catalog: {PRODUCT_CATALOG}")

In [0]:
product_master = f"{PRODUCT_CATALOG}.gold.product_master"

inventory_source = f"{INVENTORY_CATALOG}.bronze.inventory"
inventory_suppliers = f"{INVENTORY_CATALOG}.bronze.suppliers"

inventory_detail = f"{INVENTORY_CATALOG}.silver.inventory_detail" 

## Diagnóstico de duplicados en Bronze

In [0]:
spark.sql(f"""
SELECT
    'inventory' AS source_table,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT product_id) AS distinct_business_keys,
    COUNT(*) - COUNT(DISTINCT product_id) AS duplicate_business_keys
FROM {inventory_source}

UNION ALL

SELECT
    'suppliers',
    COUNT(*),
    COUNT(DISTINCT supplier_id),
    COUNT(*) - COUNT(DISTINCT supplier_id)
FROM {inventory_suppliers}
""").display()

## Construcción de Silver Inventory

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {inventory_detail} AS
WITH inventory_ranked AS (
    SELECT
        product_id,
        stock,
        supplier_id,
        last_update,
        inserted_at,
        ROW_NUMBER() OVER (
            PARTITION BY product_id
            ORDER BY
                last_update DESC,
                inserted_at DESC,
                supplier_id DESC
        ) AS row_number
    FROM {inventory_source}
    WHERE product_id IS NOT NULL
),
inventory_deduplicated AS (
    SELECT
        product_id,
        stock,
        supplier_id,
        last_update
    FROM inventory_ranked
    WHERE row_number = 1
),
suppliers_ranked AS (
    SELECT
        supplier_id,
        supplier_name,
        city,
        inserted_at,
        ROW_NUMBER() OVER (
            PARTITION BY supplier_id
            ORDER BY
                inserted_at DESC,
                supplier_name DESC,
                city DESC
        ) AS row_number
    FROM {inventory_suppliers}
    WHERE supplier_id IS NOT NULL
),
suppliers_deduplicated AS (
    SELECT
        supplier_id,
        supplier_name,
        city
    FROM suppliers_ranked
    WHERE row_number = 1
)
SELECT
    i.product_id,
    p.product_name,
    p.category,
    p.price AS product_price,
    CAST(i.stock AS INT) AS stock,
    i.supplier_id,
    INITCAP(TRIM(s.supplier_name)) AS supplier_name,
    UPPER(TRIM(s.city)) AS supplier_city,
    CAST(i.last_update AS DATE) AS last_update,
    current_timestamp() AS transformed_at
FROM inventory_deduplicated i
INNER JOIN {product_master} p
    ON i.product_id = p.product_id
LEFT JOIN suppliers_deduplicated s
    ON i.supplier_id = s.supplier_id
WHERE i.stock >= 0
  AND p.price > 0
  AND p.is_active = TRUE
""")

## Validaciones de Silver Inventory

In [0]:
spark.sql(f"""
SELECT
    COUNT(*) AS record_count,
    COUNT(DISTINCT product_id) AS distinct_products,
    COUNT(*) - COUNT(DISTINCT product_id) AS duplicate_products,
    SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS null_product_ids,
    SUM(CASE WHEN stock < 0 THEN 1 ELSE 0 END) AS negative_stock,
    SUM(CASE WHEN product_price <= 0 THEN 1 ELSE 0 END) AS invalid_prices,
    SUM(CASE WHEN supplier_id IS NULL THEN 1 ELSE 0 END) AS null_supplier_ids,
    SUM(CASE WHEN supplier_name IS NULL THEN 1 ELSE 0 END) AS null_supplier_names
FROM {inventory_detail}
""").display()

spark.sql(
    f"SELECT * FROM {inventory_detail} ORDER BY last_update DESC, product_id LIMIT 20"
).display()